# Stage 04: Localised (span-level) PCL detection (Part 3)

Utilises task2 dataset span text to get pure localised signals of where in paragh pcl is actually occuring, so the model can aggreagate these local signals and global context for more accurate paragraph-level predictions

## Imports & Dataset utilities

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers import get_linear_schedule_with_warmup

# project root (one level above notebooks)
ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

from src.data.make_dataset import build_task1_task2_with_spans, validate_span_ranges, validate_span_text_alignment, truncation_rate_by_label

RAW_TASK1 = ROOT / "data" / "raw" / "dontpatronizeme_pcl.tsv"
RAW_TASK2 = ROOT / "data" / "raw" / "dontpatronizeme_categories.tsv"
TRAIN_SPLIT = ROOT / "data" / "splits" / "train_semeval_parids-labels.csv"
DEV_SPLIT = ROOT / "data" / "splits" / "dev_semeval_parids-labels.csv"

train_df, dev_df, pcl_df, spans_df_norm = build_task1_task2_with_spans(
    raw_task1_path=RAW_TASK1,
    raw_task2_path=RAW_TASK2,
    train_split_path=TRAIN_SPLIT,
    dev_split_path=DEV_SPLIT,
)

print("train_df:", train_df.shape, "| positives:", int(train_df["label_bin"].sum()))
print("dev_df:", dev_df.shape, "| positives:", int(dev_df["label_bin"].sum()))
print("Example span_ranges:", train_df.loc[train_df["label_bin"].idxmax(), "span_ranges"] if len(train_df) else None)

/home/joshua_killa/.pyenv/versions/pcl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train_df: (8375, 8) | positives: 794
dev_df: (2094, 8) | positives: 199
Example span_ranges: [(157, 242)]


In [2]:
# Make sure span ranges are valid
validate_span_ranges(train_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="train")
validate_span_ranges(dev_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="dev")

# Check task2 spans match the corresponding substrings in task1 (only 2 mismatches and only by one charcter offset which is acceptable given token based stuff)
validate_span_text_alignment(
    train_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="train", max_mismatches=1
)
validate_span_text_alignment(
    dev_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="dev", max_mismatches=1
)


== validate_span_ranges: train ==
shape: (8375, 8)
missing (par_id,s,e) pairs: 0
out-of-bounds (par_id,s,e,L): 0
negatives with spans: 0

== validate_span_ranges: dev ==
shape: (2094, 8)
missing (par_id,s,e) pairs: 0
out-of-bounds (par_id,s,e,L): 0
negatives with spans: 0

== validate_span_text_alignment: train ==
total spans checked: 2466
text mismatches: 1
examples:


,par_id,span_start_norm,span_finish_norm,span_text,task1_substr
1134,4655,16,114,"To suffer and empathize together with others ,...","o suffer and empathize together with others , ..."



== validate_span_text_alignment: dev ==
total spans checked: 714
text mismatches: 1
examples:


,par_id,span_start_norm,span_finish_norm,span_text,task1_substr
482,6708,0,133,I only wish they can one day wake up to realis...,I only wish they can one day wake up to relise...


In [3]:
# ---- choose backbone here ----
MODEL_CANDIDATES = {
    "deberta": "microsoft/deberta-v3-base",
    "albert": "albert-base-v2",
    "albert_large": "albert-large-v2",
}

MODEL_KEY = os.getenv("PCL_BACKBONE", "albert_large")  # set to "albert" to try ALBERT
MODEL_NAME = MODEL_CANDIDATES[MODEL_KEY]
MAX_LEN = 192

def load_tokenizer(model_name: str):
    # Online first, then local cache fallback (for DNS/no-internet issues)
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print(f"Tokenizer load failed ({type(e).__name__}: {e}). Trying local cache...")
        return AutoTokenizer.from_pretrained(model_name, use_fast=True, local_files_only=True)

tokenizer = load_tokenizer(MODEL_NAME)

class PCLTokenDataset(Dataset):
    """
    Expects df columns:
      - text (str)
      - label_bin (0/1)
      - span_ranges (list[(start,end)])  (can be empty list)
    Produces:
      - input_ids, attention_mask
      - token_type_ids (only if tokenizer returns it; needed for ALBERT/BERT-like)
      - token_labels (0/1 per token)
      - token_loss_mask (bool mask for real tokens only; excludes padding + specials)
      - paragraph_label (float)
    """
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = row["text"]

        # ---- guard: tokenizer requires str / list[str] ----
        # If text is NaN/None/float, turn into empty string (or str(text) if you prefer).
        if text is None:
            text = ""
        elif isinstance(text, float):
            # catches NaN too (np.nan is float)
            if np.isnan(text):
                text = ""
            else:
                text = str(text)
        elif not isinstance(text, str):
            text = str(text)

        enc = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = enc["offset_mapping"][0]          # (T,2)
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]  # (T,)
        token_type_ids = enc["token_type_ids"][0] if "token_type_ids" in enc else None

        token_labels = torch.zeros(MAX_LEN, dtype=torch.float32)

        # real tokens: not padding AND not special tokens (specials often have offset (0,0))
        is_real_token = (attention_mask == 1) & (offsets[:, 1] > offsets[:, 0])

        spans = row["span_ranges"] if "span_ranges" in row.index else []
        if not isinstance(spans, list):
            spans = []

        # assign token label = 1 if token overlaps any annotated span
        for i, (start, end) in enumerate(offsets.tolist()):
            if not bool(is_real_token[i].item()):
                continue
            for s, e in spans:
                if start < e and end > s:
                    token_labels[i] = 1.0
                    break

        enc.pop("offset_mapping")

        batch = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_labels": token_labels,
            "token_loss_mask": is_real_token,
            "paragraph_label": torch.tensor(float(row["label_bin"]), dtype=torch.float32),
        }
        if token_type_ids is not None:
            batch["token_type_ids"] = token_type_ids

        return batch

In [4]:
# Verify max token length covers most examples (192 seems fine)
print(truncation_rate_by_label(tokenizer, train_df, 192))
print(truncation_rate_by_label(tokenizer, train_df, 200))
print(truncation_rate_by_label(tokenizer, train_df, 224))

{'max_len': 192, 'all_truncated_pct': 0.6447761194029851, 'pos_truncated_pct': 0.5037783375314862, 'neg_truncated_pct': 0.6595435958316844, 'pos_p99_len': 175, 'neg_p99_len': 180}
{'max_len': 200, 'all_truncated_pct': 0.4895522388059702, 'pos_truncated_pct': 0.5037783375314862, 'neg_truncated_pct': 0.4880622609154465, 'pos_p99_len': 175, 'neg_p99_len': 180}
{'max_len': 224, 'all_truncated_pct': 0.20298507462686569, 'pos_truncated_pct': 0.3778337531486146, 'neg_truncated_pct': 0.18467220683287164, 'pos_p99_len': 175, 'neg_p99_len': 180}


In [5]:
# How to pool token-level logits to single paragraph level logit (used in TokenCLSModel)

class LogitPooler(nn.Module):
    """
    Pool token-level logits (B,T) -> (B,1) using a boolean mask (B,T).

    Modes:
      - "max": masked max over tokens
      - "topk_mean": mean of top-k masked logits (k set by top_k)
    """
    def __init__(self, mode: str = "max", top_k: int = 3, sentinel: float = -1e4):
        super().__init__()
        self.mode = str(mode)
        self.top_k = int(top_k)
        self.sentinel = float(sentinel)

        if self.mode not in {"max", "topk_mean"}:
            raise ValueError(f"Unknown pooling mode: {self.mode}")

    def forward(self, token_logits: torch.Tensor, token_mask: torch.Tensor) -> torch.Tensor:
        """
        token_logits: (B,T) float
        token_mask:   (B,T) bool (True = keep / real tokens)
        returns:      (B,1)
        """
        token_mask = token_mask.to(dtype=torch.bool)

        # Edge-case guard: if a sample has no real tokens, return 0.0 (neutral feature)
        no_real = ~token_mask.any(dim=1, keepdim=True)  # (B,1)

        x = token_logits.masked_fill(~token_mask, self.sentinel)  # (B,T)

        if self.mode == "max":
            pooled = x.max(dim=1, keepdim=True).values  # (B,1)
            pooled = pooled.masked_fill(no_real, 0.0)
            return pooled

        # self.mode == "topk_mean"
        B, T = x.shape
        k = min(self.top_k, T)
        topk_vals = torch.topk(x, k=k, dim=1).values  # (B,k)

        # If k > #real tokens, topk will include sentinel values; exclude them from the mean.
        valid = topk_vals > (self.sentinel + 1.0)
        denom = valid.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (topk_vals.masked_fill(~valid, 0.0).sum(dim=1, keepdim=True)) / denom
        pooled = pooled.masked_fill(no_real, 0.0)
        return pooled

In [ ]:
class TokenCLSModel(nn.Module):
    def __init__(self, model_name, lambda_token=0.3, pool_mode="max", top_k=3):
        super().__init__()

        # Online first, then local cache fallback
        try:
            self.encoder = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Model load failed ({type(e).__name__}: {e}). Trying local cache...")
            self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)

        hidden = self.encoder.config.hidden_size

        self.token_head = nn.Linear(hidden, 1)
        self.paragraph_head = nn.Linear(hidden + 1, 1)

        self.lambda_token = float(lambda_token)

        # modular pooling (swap "max" <-> "topk_mean")
        self.pooler = LogitPooler(mode=pool_mode, top_k=top_k)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        token_labels=None,
        token_loss_mask=None,
        paragraph_label=None,
    ):
        # Some backbones ignore token_type_ids; some accept it.
        # Try passing it; if unsupported, fall back.
        try:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        hidden = outputs.last_hidden_state  # (B,T,H)

        # keep dtype consistent with heads (prevents dtype mismatch issues)
        hidden = hidden.to(dtype=self.token_head.weight.dtype)

        cls = hidden[:, 0]  # (B,H)
        token_logits = self.token_head(hidden).squeeze(-1)  # (B,T)

        # safe token mask for pooling + token-loss
        if token_loss_mask is None:
            token_loss_mask = attention_mask == 1
        token_loss_mask = token_loss_mask.to(dtype=torch.bool)

        pooled = self.pooler(token_logits, token_loss_mask)  # (B,1)

        fused = torch.cat([cls, pooled], dim=1)  # (B,H+1)
        paragraph_logit = self.paragraph_head(fused).squeeze(-1)  # (B,)

        loss = None
        loss_par = None
        loss_tok = None

        if paragraph_label is not None:
            loss_par = F.binary_cross_entropy_with_logits(
                paragraph_logit.float(),
                paragraph_label.float(),
            )

            if token_loss_mask is None:
                token_loss_mask = attention_mask == 1
            token_loss_mask = token_loss_mask.to(dtype=torch.bool)

            pos_rows = (paragraph_label > 0.5)
            neg_rows = ~pos_rows

            tok_mask_pos = token_loss_mask & pos_rows.unsqueeze(1)
            tok_mask_neg = token_loss_mask & neg_rows.unsqueeze(1)

            loss_tok_pos = torch.zeros((), device=token_logits.device, dtype=torch.float32)
            if tok_mask_pos.any():
                loss_tok_pos = F.binary_cross_entropy_with_logits(
                    token_logits[tok_mask_pos].float(),
                    token_labels[tok_mask_pos].float(),
                )

            loss_tok_neg = torch.zeros((), device=token_logits.device, dtype=torch.float32)
            if tok_mask_neg.any():
                loss_tok_neg = F.binary_cross_entropy_with_logits(
                    token_logits[tok_mask_neg].float(),
                    token_labels[tok_mask_neg].float(),
                )

            alpha_neg = 0.25  # try 0.1, 0.25, 0.5
            loss_tok = loss_tok_pos + alpha_neg * loss_tok_neg

            loss = loss_par + self.lambda_token * loss_tok

        return {
            "loss": loss,
            "loss_par": loss_par,
            "loss_tok": loss_tok,
            "paragraph_logit": paragraph_logit,
            "token_logits": token_logits,
        }

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _amp_settings(model_key: str, model_name: str):
    key = (model_key or "").lower()
    name = (model_name or "").lower()

    if ("albert" in key) or ("albert" in name):
        enabled = torch.cuda.is_available()
        return enabled, torch.float16, True  # FP16 + GradScaler

    if ("deberta" in key) or ("deberta" in name):
        enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        return enabled, torch.bfloat16, False  # BF16, no GradScaler

    return False, None, False

USE_AMP, AMP_DTYPE, NEEDS_SCALER = _amp_settings(MODEL_KEY, MODEL_NAME)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

print(
    f"DEVICE={DEVICE} | backbone={MODEL_KEY} | amp={USE_AMP} | amp_dtype={AMP_DTYPE} | "
    f"scaler={bool(scaler.is_enabled())}"
)

model = TokenCLSModel(MODEL_NAME, lambda_token=0.3, pool_mode="max").to(DEVICE)

train_dataset = PCLTokenDataset(train_df)

# ---- balanced paragraph sampling (approx 50/50 pos/neg) ----
labels = train_df["label_bin"].astype(int).to_numpy()
pos = int(labels.sum())
neg = int(len(labels) - pos)

w_pos = 1.0 / max(pos, 1)
w_neg = 1.0 / max(neg, 1)

sample_weights = torch.tensor([w_pos if y == 1 else w_neg for y in labels], dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # keep "epoch" size comparable to dataset size
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,   # <- sampler replaces shuffle
)

# separate loaders for F1 computation (no shuffle)
train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
dev_dataset = PCLTokenDataset(dev_df)
dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

optimizer = AdamW(
    model.parameters(),
    lr=7e-6,
    weight_decay=1e-2,
)

THRESH = 0.5
GRAD_ACCUM = 2  # <- match your optuna best configs; effective batch = 16 * 2 = 32
EPOCHS = 7

total_update_steps = (len(train_loader) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
warmup_steps = int(0.07 * total_update_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)

def _bin_metrics_from_logits(logits: np.ndarray, labels: np.ndarray, threshold: float = 0.5):
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= threshold).astype(int)
    y = labels.astype(int)

    tp = int(((preds == 1) & (y == 1)).sum())
    tn = int(((preds == 0) & (y == 0)).sum())
    fp = int(((preds == 1) & (y == 0)).sum())
    fn = int(((preds == 0) & (y == 1)).sum())

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    acc_nonpcl = tn / max(tn + fp, 1)   # specificity
    acc_pcl    = tp / max(tp + fn, 1)   # recall for positives

    return {
        "acc": float(acc),
        "acc_nonpcl": float(acc_nonpcl),
        "acc_pcl": float(acc_pcl),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }

def stats_on_loader(loader, threshold: float = 0.5):
    model.eval()
    all_logits, all_labels = [], []
    losses, losses_par, losses_tok = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    out = model(**batch)
            else:
                out = model(**batch)

            # logits/labels for metrics
            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

            # losses (already computed in fp32 inside your forward)
            if out.get("loss") is not None:
                losses.append(float(out["loss"].detach().float().cpu()))
            if out.get("loss_par") is not None:
                losses_par.append(float(out["loss_par"].detach().float().cpu()))
            if out.get("loss_tok") is not None:
                losses_tok.append(float(out["loss_tok"].detach().float().cpu()))

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    m = _bin_metrics_from_logits(logits, labels, threshold=threshold)
    m["f1"] = float(f1_score(labels, (1.0 / (1.0 + np.exp(-logits)) >= threshold).astype(int), pos_label=1))

    m["loss"] = float(np.mean(losses)) if losses else None
    m["loss_par"] = float(np.mean(losses_par)) if losses_par else None
    m["loss_tok"] = float(np.mean(losses_tok)) if losses_tok else None
    return m

def f1_on_loader(loader) -> float:
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    out = model(**batch)
            else:
                out = model(**batch)

            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs > THRESH).astype(int)
    return f1_score(labels, preds, pos_label=1)

DEVICE=cuda | backbone=albert_large | amp=True | amp_dtype=torch.float16 | scaler=True


/tmp/ipykernel_155004/472945824.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 387.36it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
b = next(iter(train_loader))
print("batch pos rate:", float(b["paragraph_label"].mean()))

batch pos rate: 0.5


In [ ]:
# Optional curriculum: token-only warm start
TOKEN_ONLY_EPOCHS = 1
for p in model.paragraph_head.parameters():
    p.requires_grad = True  # default

for epoch in range(EPOCHS):
    # ---- curriculum toggle ----
    if epoch < TOKEN_ONLY_EPOCHS:
        for p in model.paragraph_head.parameters():
            p.requires_grad = False
        model.lambda_token = 1.0
    else:
        for p in model.paragraph_head.parameters():
            p.requires_grad = True
        model.lambda_token = 0.3

    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(train_loader), start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                out = model(**batch)
                loss = out["loss"]
        else:
            out = model(**batch)
            loss = out["loss"]

        if torch.isnan(loss) or torch.isinf(loss):
            raise RuntimeError("Loss became NaN/Inf. Inspect batch / masks / logits.")

        loss_to_backprop = loss / GRAD_ACCUM

        if scaler.is_enabled():
            scaler.scale(loss_to_backprop).backward()
        else:
            loss_to_backprop.backward()

        total_loss += float(loss.detach().float().cpu())

        if (step % GRAD_ACCUM) == 0:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    train_stats = stats_on_loader(train_eval_loader, threshold=THRESH)
    dev_stats = stats_on_loader(dev_eval_loader, threshold=THRESH)

    print(
        f"Epoch {epoch} | "
        f"train loss={train_stats['loss']:.4f} (par={train_stats['loss_par']:.4f}, tok={train_stats['loss_tok']:.4f}) | "
        f"train f1={train_stats['f1']:.4f} acc={train_stats['acc']:.4f} acc0={train_stats['acc_nonpcl']:.4f} acc1={train_stats['acc_pcl']:.4f} | "
        f"dev loss={dev_stats['loss']:.4f} (par={dev_stats['loss_par']:.4f}, tok={dev_stats['loss_tok']:.4f}) | "
        f"dev f1={dev_stats['f1']:.4f} acc={dev_stats['acc']:.4f} acc0={dev_stats['acc_nonpcl']:.4f} acc1={dev_stats['acc_pcl']:.4f} | "
        f"lambda_token={model.lambda_token} | "
        f"backbone={MODEL_KEY} | amp={USE_AMP} dtype={AMP_DTYPE}"
    )

100%|██████████| 524/524 [06:16<00:00,  1.39it/s]


Epoch 0 | Loss: 0.5933 | train_f1@0.5: 0.6883 | dev_f1@0.5: 0.5424 | backbone=albert_large | amp=True | dtype=torch.float16


100%|██████████| 524/524 [06:46<00:00,  1.29it/s]


Epoch 1 | Loss: 0.3252 | train_f1@0.5: 0.8250 | dev_f1@0.5: 0.4746 | backbone=albert_large | amp=True | dtype=torch.float16


100%|██████████| 524/524 [06:48<00:00,  1.28it/s]


...

Epoch 4 | Loss: 0.0870 | train_f1@0.5: 0.9457 | dev_f1@0.5: 0.4173 | backbone=albert_large | amp=True | dtype=torch.float16


100%|██████████| 524/524 [07:01<00:00,  1.24it/s]


Epoch 5 | Loss: 0.0577 | train_f1@0.5: 0.9670 | dev_f1@0.5: 0.4786 | backbone=albert_large | amp=True | dtype=torch.float16


100%|██████████| 524/524 [06:52<00:00,  1.27it/s]


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import numpy as np

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=32)
    model.eval()

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    probs = 1.0 / (1.0 + np.exp(-logits))

    best_f1 = 0.0
    best_thresh = 0.5
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = float(t)

    return best_f1, best_thresh

In [ ]:
# #empty torch cahce

# torch.cuda.empty_cache()

In [11]:
def _merge_ranges(ranges):
    """ranges: list of (start,end) half-open char intervals. returns merged non-overlapping list."""
    if not ranges:
        return []
    ranges = [(int(s), int(e)) for s, e in ranges if s is not None and e is not None and int(e) > int(s)]
    if not ranges:
        return []
    ranges.sort(key=lambda x: (x[0], x[1]))
    merged = [list(ranges[0])]
    for s, e in ranges[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e:  # overlap or touch
            merged[-1][1] = max(last_e, e)
        else:
            merged.append([s, e])
    return [(s, e) for s, e in merged]

def span_coverage_char_level(df, text_col="text", spans_col="span_ranges"):
    total_chars = 0
    total_span_chars = 0
    n_rows = len(df)

    for _, row in df.iterrows():
        text = row.get(text_col, "")
        if text is None or (isinstance(text, float) and np.isnan(text)):
            text = ""
        if not isinstance(text, str):
            text = str(text)

        L = len(text)
        total_chars += L

        spans = row.get(spans_col, [])
        if not isinstance(spans, list):
            spans = []

        merged = _merge_ranges(spans)
        merged = [(max(0, s), min(L, e)) for s, e in merged if min(L, e) > max(0, s)]
        total_span_chars += sum(e - s for s, e in merged)

    span_frac = (total_span_chars / total_chars) if total_chars > 0 else 0.0
    return {
        "rows": int(n_rows),
        "total_chars": int(total_chars),
        "span_chars": int(total_span_chars),
        "nonspan_chars": int(total_chars - total_span_chars),
        "span_frac": float(span_frac),
    }

def per_row_span_frac(df, text_col="text", spans_col="span_ranges"):
    """Returns an array with per-row fraction of characters covered by merged spans."""
    fracs = []
    for _, row in df.iterrows():
        text = row.get(text_col, "")
        if text is None or (isinstance(text, float) and np.isnan(text)):
            text = ""
        if not isinstance(text, str):
            text = str(text)

        L = len(text)
        if L <= 0:
            fracs.append(0.0)
            continue

        spans = row.get(spans_col, [])
        if not isinstance(spans, list):
            spans = []

        merged = _merge_ranges(spans)
        merged = [(max(0, s), min(L, e)) for s, e in merged if min(L, e) > max(0, s)]
        span_chars = sum(e - s for s, e in merged)
        fracs.append(span_chars / L)
    return np.asarray(fracs, dtype=np.float64)

def paragraph_balance(df, label_col="label_bin"):
    pos = int(df[label_col].sum())
    neg = int(len(df) - pos)
    pos_rate = pos / len(df) if len(df) else 0.0
    pos_weight = (neg / pos) if pos > 0 else None
    return {"rows": int(len(df)), "pos": pos, "neg": neg, "pos_rate": float(pos_rate), "pos_weight": pos_weight}

def span_stats_report(df, name="df", label_col="label_bin", text_col="text", spans_col="span_ranges"):
    df_pos = df[df[label_col].astype(int) == 1]
    df_neg = df[df[label_col].astype(int) == 0]

    overall_cov = span_coverage_char_level(df, text_col=text_col, spans_col=spans_col)
    pos_cov = span_coverage_char_level(df_pos, text_col=text_col, spans_col=spans_col)
    neg_cov = span_coverage_char_level(df_neg, text_col=text_col, spans_col=spans_col)

    pos_row_fracs = per_row_span_frac(df_pos, text_col=text_col, spans_col=spans_col) if len(df_pos) else np.array([])
    neg_row_fracs = per_row_span_frac(df_neg, text_col=text_col, spans_col=spans_col) if len(df_neg) else np.array([])

    print(f"\n== {name} ==")
    print("Paragraph balance:", paragraph_balance(df, label_col=label_col))
    print("Span coverage (overall):", overall_cov)
    print("Span coverage (positives only):", pos_cov)
    print("Span coverage (negatives only):", neg_cov)

    # Per-row averages (answers: "average fraction of span for positive classes only")
    if len(pos_row_fracs):
        print(
            "Pos-only per-row span_frac: "
            f"mean={pos_row_fracs.mean():.6f} | median={np.median(pos_row_fracs):.6f} | "
            f"p90={np.quantile(pos_row_fracs, 0.90):.6f}"
        )
    else:
        print("Pos-only per-row span_frac: n/a (no positive rows)")

    if len(neg_row_fracs):
        print(
            "Neg-only per-row span_frac: "
            f"mean={neg_row_fracs.mean():.6f} | median={np.median(neg_row_fracs):.6f} | "
            f"p90={np.quantile(neg_row_fracs, 0.90):.6f}"
        )
    else:
        print("Neg-only per-row span_frac: n/a (no negative rows)")

    # Approx token pos_weight from char coverage (overall + pos-only)
    if overall_cov["span_chars"] > 0:
        print("Approx token pos_weight (overall char nonspan/span):", float(overall_cov["nonspan_chars"] / overall_cov["span_chars"]))
    else:
        print("Approx token pos_weight (overall): None (no span chars?)")

    if pos_cov["span_chars"] > 0:
        print("Approx token pos_weight (pos-only char nonspan/span):", float(pos_cov["nonspan_chars"] / pos_cov["span_chars"]))
    else:
        print("Approx token pos_weight (pos-only): None (no span chars in positives?)")

span_stats_report(train_df, name="train_df")
span_stats_report(dev_df, name="dev_df")


== train_df ==
Paragraph balance: {'rows': 8375, 'pos': 794, 'neg': 7581, 'pos_rate': 0.09480597014925374, 'pos_weight': 9.547858942065492}
Span coverage (overall): {'rows': 8375, 'total_chars': 2243329, 'span_chars': 145477, 'nonspan_chars': 2097852, 'span_frac': 0.06484871367507843}
Span coverage (positives only): {'rows': 794, 'total_chars': 227501, 'span_chars': 145477, 'nonspan_chars': 82024, 'span_frac': 0.6394565298614072}
Span coverage (negatives only): {'rows': 7581, 'total_chars': 2015828, 'span_chars': 0, 'nonspan_chars': 2015828, 'span_frac': 0.0}
Pos-only per-row span_frac: mean=0.693176 | median=0.739633 | p90=0.987998
Neg-only per-row span_frac: mean=0.000000 | median=0.000000 | p90=0.000000
Approx token pos_weight (overall char nonspan/span): 14.420506334334638
Approx token pos_weight (pos-only char nonspan/span): 0.5638279590588203

== dev_df ==
Paragraph balance: {'rows': 2094, 'pos': 199, 'neg': 1895, 'pos_rate': 0.09503342884431709, 'pos_weight': 9.522613065326633}